In [6]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv
import math 

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] 
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
con.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")


print("✅ DuckDB의 MinIO 접속 준비 완료!")



✅ DuckDB의 MinIO 접속 준비 완료!


In [4]:

df = con.sql(
    """
    select 
    uni_cd
    , gasoline
    , diesel
    , premium_gasoline
    from read_parquet('s3://petroleum-project/station_price/agg/*/*.parquet')
    where gasoline is not null and diesel is not null and premium_gasoline is not null  -- 첫번째 샘플은 걍 모든 데이터 있는 것으로.
    and part_dt = '20260425'
    order by uni_cd
    limit 2

"""
).df()

display(df)

,uni_cd,gasoline,diesel,premium_gasoline
0,A0000004,2009,2001,2388
1,A0000011,1984,1984,2385


In [5]:
# 체득 1 모든 과정을 직접 계산하기

i = 0
A = df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]
B = df.iloc[1][['gasoline' , 'diesel' , 'premium_gasoline']]

dot = (A*B).sum()
display(dot)

13651220

In [12]:
# dot product는 수기로 계산했으니, 각각의 거리를 계산하자.
len_a = math.sqrt(df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]**2)
#len_a = math.sqrt( df.iloc[0][['gasoline' , 'diesel' , 'premium_gasoline']]**2)
display(len_a)

TypeError: cannot convert the series to <class 'float'>